In [13]:
import math
import numpy as np

Building from the micrograd I created a basic autograd engine that works with tensors.

A large proportion of the structure remains the same or very similar. Basic operations such as addition, multiplication, subtraction, powers and division largely remain the same with the addition of handling broadcasting for grads.

Several features are required for use of tensors rather than using scalars:
- Tensor multiplication (in terms of batch matrix multiplication) must be introduced and hence the grads as well
- Handing broadcasting and in particular how grads are calculated

In addition to the above I introduced other useful operations commonly used in neural networks, which were all straightforward to implement:
- reshaping
- transposing for general tensors (swapping the last 2 axis)
- ReLU
- softmax
- logsoftmax
- max
- log
- sums
- means

We take advantage of the nature of the operations we do to form compact representations of the jacobian. For example in addition, if Y=X+Z, only the (i,j) entry of X and Z affect the (i,j) entry of Y hence we don't need to even think about the other entries (which will have grad 0).

To handle broadcasting we can just think of a mapping from the original to a broadcasted version which is used in the next calculation. We can then use backpropagation with the chain rule to calculate the grad.

In [14]:
import numpy as np

class Tensor:
    def __init__(self,data,prev=(),requires_grad=False):
        self.data=np.asarray(data, dtype=np.float64)
        self.parent=tuple(prev)
        self.requires_grad=requires_grad
        self.grad=np.zeros_like(self.data) if requires_grad else None
        self._backward=lambda: None
        self.shape=np.shape(self.data)
    
    def __add__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        out=Tensor(self.data+other.data,(self,other),requires_grad=self.requires_grad or other.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=Tensor.unbroadcast(out.grad,self.shape)
            if other.requires_grad:
                other.grad+=Tensor.unbroadcast(out.grad,other.shape)
        out._backward=_backward
        return out

    def __mul__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        out=Tensor(self.data*other.data,(self,other),requires_grad=self.requires_grad or other.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=Tensor.unbroadcast(other.data*out.grad,self.shape)
            if other.requires_grad:
                other.grad+=Tensor.unbroadcast(self.data*out.grad,other.shape)
        out._backward=_backward
        return out

    def __pow__(self,power):
        out=Tensor(self.data**power,(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=power*(self.data**(power-1))*out.grad
        out._backward=_backward
        return out
    
    def __truediv__(self,other):
        return self*(other**-1)
    
    def __neg__(self):
        return self*(-1)
    
    def __sub__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return self+-other
    
    def __radd__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return other+self
        
    def __rsub__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return -other+self

    def __rmul__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return other*self
    
    def exp(self):
        out=Tensor(np.exp(self.data),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=out.data*out.grad
        out._backward=_backward
        return out
    
    def backward(self,grad=None):
        topo=[]
        visited=set()
        def topo_build(v):
            if v not in visited:
                visited.add(v)
                for parent in v.parent:
                    topo_build(parent)
                topo.append(v)
        topo_build(self)
        for node in topo:
            if node.requires_grad:
                node.zero_grad()
        if grad is None:
            self.grad=np.ones_like(self.data)
        else:
            self.grad=grad
        for node in reversed(topo):
            node._backward()

    def __rtruediv__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return other/self

    def zero_grad(self):
        if self.requires_grad:
            self.grad=np.zeros_like(self.data)
    
    def sum(self,axis=None,keepdims=False):
        out=Tensor(self.data.sum(axis=axis,keepdims=keepdims),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                grad=out.grad
                if axis is None:
                    grad=np.ones_like(self.data)*grad
                else:
                    if not keepdims:
                        #Adds on additional axis to grad if axis is removed when summing
                        grad=np.expand_dims(grad,axis)
                    #Broadcasts the grad to the shape of self
                    grad=np.broadcast_to(grad,self.data.shape)
                self.grad+=grad
        out._backward=_backward
        return out

    def mean(self,axis=None,keepdims=False):
        if axis is None:
            n=self.data.size
        else:
            n=self.data.shape[axis]
        return self.sum(axis=axis,keepdims=keepdims)/n
    
    @staticmethod
    def unbroadcast(grad,shape):
        #We repeatedly sum over the first axis until we have the same shape as the desired shape 
        #Numpy always adds missing dimensions to the left when broadcasting hence we sum over axis=0
        while len(grad.shape)>len(shape):
            grad=grad.sum(axis=0)
        #We then collapse all axis which are supposed to be dimension 1
        for i,dim in enumerate(shape):
            if dim==1 and grad.shape[i]!=1:
                grad=grad.sum(axis=i,keepdims=True)
        return grad
    
    def reshape(self, shape):
        out=Tensor(self.data.reshape(shape),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=out.grad.reshape(self.shape)
        out._backward=_backward
        return out
    
    def transpose(self,axis1=-2,axis2=-1):
        out=Tensor(np.swapaxes(self.data,axis1,axis2),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=np.swapaxes(out.grad,axis1,axis2)
        out._backward=_backward
        return out
    
    def __matmul__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        out=Tensor(self.data@other.data,(self,other),requires_grad=self.requires_grad or other.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=Tensor.unbroadcast(out.grad@np.swapaxes(other.data,-2,-1),self.shape)
            if other.requires_grad:
                other.grad+=Tensor.unbroadcast(np.swapaxes(self.data,-2,-1)@out.grad,other.shape)
        out._backward=_backward
        return out
    
    def log(self):
        out=Tensor(np.log(self.data),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=out.grad/self.data
        out._backward=_backward
        return out
    
    def ReLU(self):
        #self.data>0 is a compact representation of the jacobian
        out=Tensor(np.maximum(0,self.data),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                self.grad+=(self.data>0)*out.grad
        out._backward=_backward
        return out
    
    def max(self,axis=None,keepdims=False):
        #In max, mask is the compact representation of the jacobian. 
        out=Tensor(np.max(self.data, axis=axis, keepdims=keepdims),(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                grad=out.grad
                if axis is None:
                    mask=(self.data==np.max(self.data))
                    self.grad+=mask*grad
                else:
                    if not keepdims:
                        grad=np.expand_dims(grad,axis)
                    mask=(self.data==np.max(self.data,axis=axis,keepdims=True))
                    self.grad+=mask*grad
        out._backward=_backward
        return out
    
    def softmax(self,axis=-1):
        shifted=self-self.max(axis=axis,keepdims=True)
        exp=shifted.exp()
        return exp/exp.sum(axis=axis,keepdims=True)
    
    def __rmatmul__(self,other):
        if not isinstance(other,Tensor):
            other=Tensor(other)
        return other@self
    
    def log_softmax(self,axis=-1):
        shifted=self-self.max(axis=axis,keepdims=True)
        return shifted-shifted.exp().sum(axis=axis,keepdims=True).log()
    
    def __getitem__(self, idx):
        out = Tensor(self.data[idx],(self,),requires_grad=self.requires_grad)
        def _backward():
            if self.requires_grad:
                #If a single index is repeated in multiple entries the previous accumulating approaches won't work
                np.add.at(self.grad,idx,out.grad) 
        out._backward = _backward
        return out

All operations are checked below in terms of getting the correct output and correct grad on backward propagation. The grad is found through the finite differences method. This was created using gen-AI.

In [15]:
import numpy as np

def numerical_grad(fn,x,eps=1e-6):
    grad=np.zeros_like(x)
    iterator=np.nditer(x,flags=["multi_index"],op_flags=["readwrite"])
    while not iterator.finished:
        idx=iterator.multi_index
        old=x[idx]
        x[idx]=old+eps
        y1=fn(x)
        x[idx]=old-eps
        y2=fn(x)
        x[idx]=old
        grad[idx]=(y1-y2)/(2*eps)
        iterator.iternext()
    return grad

def check_grad(name,fn,x,atol=1e-5,rtol=1e-5):
    x=np.asarray(x,dtype=np.float64)
    tensor=Tensor(x.copy(),requires_grad=True)
    out=fn(tensor)
    if out.data.size!=1:
        out=out.sum()
    out.backward()
    analytic=tensor.grad.copy()
    numeric=numerical_grad(lambda z:fn(Tensor(z)).data.sum(),x.copy())
    error=np.max(np.abs(analytic-numeric))
    passed=np.allclose(analytic,numeric,atol=atol,rtol=rtol)
    print(f"{'PASS' if passed else 'FAIL'} "f"{name:<35} "f"error={error:.3e}")
    if not passed:
        print("analytic:")
        print(analytic)
        print("numeric:")
        print(numeric)
    return passed
np.random.seed(42)
passed=0
failed=0
def run(name,fn,x):
    global passed,failed

    if check_grad(name,fn,x):
        passed+=1
    else:
        failed+=1
# Basic operations
run("add scalar",lambda x:x+3,np.random.randn(5))
run("multiply scalar",lambda x:x*4,np.random.randn(5))
run("subtract",lambda x:x-2,np.random.randn(5))
run("divide scalar",lambda x:x/3,np.random.randn(5)+2)
run("power",lambda x:x**3,np.random.randn(5))
# Unary functions
run("exp",lambda x:x.exp(),np.random.randn(5))
run("log",lambda x:x.log(),np.random.rand(5)+1)
run("relu",lambda x:x.ReLU(),np.random.randn(10))
# Broadcasting tests
c=Tensor(2.0)
run("broadcast scalar add",lambda x:x+c,np.random.randn(3,4))
c=Tensor(np.random.randn(4))
run("broadcast vector",lambda x:x+c,np.random.randn(3,4))
c=Tensor(np.random.randn(1,4))
run("broadcast row vector",lambda x:x+c,np.random.randn(3,4))
c=Tensor(np.random.randn(3,1))
run("broadcast column vector",lambda x:x+c,np.random.randn(3,4))
c=Tensor(np.random.randn(1,3,1))
run("broadcast higher rank",lambda x:x+c,np.random.randn(2,3,4))
# Multiplication broadcasting
c=Tensor(np.random.randn(5))
run("multiply broadcast vector",lambda x:x*c,np.random.randn(2,3,5))
c=Tensor(np.random.randn(1,4,1))
run("multiply middle broadcast",lambda x:x*c,np.random.randn(2,4,5))
# Reduction operations
x=np.random.randn(3,4,5)
run("sum all",lambda x:x.sum(),x)
run("sum axis0",lambda x:x.sum(axis=0),x)
run("sum axis1",lambda x:x.sum(axis=1),x)
run("sum keepdims",lambda x:x.sum(axis=1,keepdims=True),x)
run("mean",lambda x:x.mean(),x)
run("mean axis",lambda x:x.mean(axis=2),x)
# Shape operations
run("reshape",lambda x:x.reshape((6,4)),np.random.randn(2,3,4))
run("swap last axes",lambda x:x.transpose(),np.random.randn(3,4))
# Matrix multiplication
W=Tensor(np.random.randn(5,6))
run("matrix multiply",lambda x:x@W,np.random.randn(4,5))
W=Tensor(np.random.randn(5,6))
run("vector matrix multiply",lambda x:x@W,np.random.randn(5))
W=Tensor(np.random.randn(2,5,6))
run("batched matmul",lambda x:x@W,np.random.randn(2,4,5))
W=Tensor(np.random.randn(1,5,6))
run("broadcast batch matmul",lambda x:x@W,np.random.randn(3,4,5))
# Softmax
run("softmax",lambda x:x.softmax(),np.random.randn(3,5))
run("softmax axis",lambda x:x.softmax(axis=1),np.random.randn(2,3,5))
run("log softmax",lambda x:x.log_softmax(),np.random.randn(4,6))
# Random rank stress tests
for rank in range(1,6):
    shape=tuple(np.random.randint(2,5) for _ in range(rank))
    run(f"random rank {rank}",lambda x:(x*x.exp()).sum(),np.random.randn(*shape))
print("\n======================")
print("Passed:",passed)
print("Failed:",failed)
print("======================")

PASS add scalar                          error=1.028e-09
PASS multiply scalar                     error=5.591e-10
PASS subtract                            error=7.484e-10
PASS divide scalar                       error=1.755e-10
PASS power                               error=4.988e-10
PASS exp                                 error=3.253e-10
PASS log                                 error=1.079e-10
PASS relu                                error=3.043e-10
PASS broadcast scalar add                error=1.028e-09
PASS broadcast vector                    error=1.398e-10
PASS broadcast row vector                error=3.043e-10
PASS broadcast column vector             error=3.043e-10
PASS broadcast higher rank               error=1.028e-09
PASS multiply broadcast vector           error=3.231e-10
PASS multiply middle broadcast           error=2.877e-10
PASS sum all                             error=7.484e-10
PASS sum axis0                           error=7.484e-10
PASS sum axis1                 